In [15]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv(override=True)
google_api_key = os.getenv('GOOGLE_API_KEY')
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)


In [8]:
# 1. Define the actual local dummy functions
def record_positive_adjectives(figure_name: str, positive_adjectives: list[str]) -> str:
    print(f"Recording positive adjectives for {figure_name}: {positive_adjectives}")
    adjectives_str = ", ".join(positive_adjectives)
    return f"[Positive Attributes Recorded for {figure_name}]: {adjectives_str}"

def record_critical_adjectives(figure_name: str, critical_adjectives: list[str]) -> str:
    print(f"Recording critical adjectives for {figure_name}: {critical_adjectives}")
    adjectives_str = ", ".join(critical_adjectives)
    return f"[Critical/Flawed Attributes Recorded for {figure_name}]: {adjectives_str}"

In [9]:
# 2. Tool schemas for the LLM
tools = [
    {
        "type": "function",
        "function": {
            "name": "record_positive_adjectives",
            "description": "Records commendable, virtuous, or positive adjectives describing a historical figure.",
            "parameters": {
                "type": "object",
                "properties": {
                    "figure_name": {"type": "string", "description": "Name of the historical figure"},
                    "positive_adjectives": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "List of positive adjectives describing this person"
                    }
                },
                "required": ["figure_name", "positive_adjectives"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "record_critical_adjectives",
            "description": "Records negative, flawed, controversial, or critical adjectives describing a historical figure.",
            "parameters": {
                "type": "object",
                "properties": {
                    "figure_name": {"type": "string", "description": "Name of the historical figure"},
                    "critical_adjectives": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "List of negative or critical adjectives describing this person"
                    }
                },
                "required": ["figure_name", "critical_adjectives"]
            }
        }
    }
]

# Mapping tool names to local functions
tool_map = {
    "record_positive_adjectives": record_positive_adjectives,
    "record_critical_adjectives": record_critical_adjectives
}

In [13]:


# 3. Main execution function
def analyze_historical_figure(figure_query: str):
    messages = [
        {
            "role": "system",
            "content": "You are a historical analyst. When asked about a historical figure, ALWAYS call both available tools: one to record their positive adjectives and one to record their critical/flawed adjectives. Then summarize your verdict using those results."
        },
        {"role": "user", "content": figure_query}
    ]

    # First LLM call: LLM decides which tools to call
    response = gemini.chat.completions.create(
        model="gemini-3.1-flash-lite",
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )
    
    response_message = response.choices[0].message
    tool_calls = response_message.tool_calls

    if not tool_calls:
        return response_message.content

    # Append assistant's intent to call tools
    messages.append(response_message)

    # Execute each tool call and return result to messages
    for tool_call in tool_calls:
        fn_name = tool_call.function.name
        fn_args = json.loads(tool_call.function.arguments)
        
        target_fn = tool_map.get(fn_name)
        tool_result = target_fn(**fn_args)

        messages.append({
            "tool_call_id": tool_call.id,
            "role": "tool",
            "name": fn_name,
            "content": tool_result
        })

    # Second LLM call: synthesize final response with tool outputs included
    final_response = gemini.chat.completions.create(
        model="gemini-3.1-flash-lite",
        messages=messages
    )

    return final_response.choices[0].message.content

In [16]:
# Test
display(Markdown(analyze_historical_figure("Tell me about Napoleon Bonaparte.")))

Recording positive adjectives for Napoleon Bonaparte: ['brilliant', 'charismatic', 'visionary', 'decisive', 'diligent', 'inspirational']
Recording critical adjectives for Napoleon Bonaparte: ['authoritarian', 'megalomaniacal', 'belligerent', 'ruthless', 'imperialistic', 'oppressive']


Napoleon Bonaparte remains one of the most polarizing figures in modern history, a complex individual whose legacy is defined by a sharp dichotomy between his transformative governance and his destructive ambitions.

**The Verdict:**

Napoleon was a **brilliant, charismatic, and visionary** leader whose administrative reforms—such as the Napoleonic Code—left an indelible mark on legal and civil structures across Europe. He was a **decisive and diligent** strategist who inspired intense loyalty among his troops, effectively reshaping the map of the continent through his tactical genius.

However, these accomplishments were shadowed by a deeply **authoritarian and megalomaniacal** nature. His **belligerent** foreign policy and **imperialistic** drive triggered decades of devastating continental warfare. While he viewed himself as a modernizer, his methods were frequently **ruthless and oppressive**, sacrificing countless lives to satisfy his pursuit of global hegemony. Ultimately, Napoleon stands as a quintessential example of how transformative ambition can simultaneously build institutional foundations while creating immense human suffering.

In [17]:
import gradio as gr

def gradio_chat(figure_name):
    if not figure_name.strip():
        return "Please enter a historical figure's name."
    
    query = f"Analyze the historical figure: {figure_name}"
    return analyze_historical_figure(query)

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🏛️ Historical Figure Adjective Tool Calling")
    gr.Markdown("Enter any historical figure. The LLM will trigger two tools (`record_positive_adjectives` & `record_critical_adjectives`) and return a balanced appraisal.")
    
    with gr.Row():
        input_text = gr.Textbox(
            label="Historical Figure",
            placeholder="e.g., Julius Caesar, Winston Churchill, Cleopatra...",
            lines=1
        )
        submit_btn = gr.Button("Analyze", variant="primary")
        
    output_text = gr.Textbox(
        label="LLM Synthesis (via Tool Execution)",
        lines=10
    )
    
    submit_btn.click(fn=gradio_chat, inputs=input_text, outputs=output_text)
    input_text.submit(fn=gradio_chat, inputs=input_text, outputs=output_text)

# Launch in Jupyter Notebook
demo.launch(share=False)

C:\Users\justa\AppData\Local\Temp\ipykernel_16428\1026777420.py:10: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


c:\Amit\code\AI\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Recording positive adjectives for Adolf Hitler: ['charismatic', 'persuasive', 'oratorical', 'determined']
Recording critical adjectives for Adolf Hitler: ['genocidal', 'authoritarian,', 'megalomaniacal,', 'hateful,', 'tyrannical,', 'destructive,', 'evil,', 'racist,', 'antisemitic,']


c:\Amit\code\AI\agents\.venv\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Recording positive adjectives for Stormy Daniels: ['resilient', 'articulate', 'candid', 'unapologetic']
Recording critical adjectives for Stormy Daniels: ['controversial', 'polarizing']
